# Robustness Tests 

Author: Gabriel Geiger 
Date: 23-03-2026 

Runs a series of robustness tests. 

In [33]:
import sys 
from pathlib import Path
import pandas as pd 

import statsmodels.api as sm
import numpy as np

# Import project modules
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

from config import paths 

#### Load Validation dataframes 

In [34]:
kchs_df = pd.read_csv(paths.PROCESSED_DATA / 'kchs_2021_validation.csv', low_memory=False)
ibs_df = pd.read_csv(paths.PROCESSED_DATA / 'ibs_2015_validation.csv', low_memory=False)

# Add an excluded variable 
kchs_df['excluded']= (kchs_df['Predicted_Poor'] == False) & (kchs_df['Actually_Poor'] == True)
ibs_df['excluded']= (ibs_df['Predicted_Poor'] == False) & (ibs_df['Actually_Poor'] == True)

kchs_rural = kchs_df[kchs_df['urban_rural_classification'] == 'Rural']
kchs_urban = kchs_df[kchs_df['urban_rural_classification'] == 'Urban']
ibs_rural = ibs_df[ibs_df['urban_rural_classification'] == 'Rural']
ibs_urban = ibs_df[ibs_df['urban_rural_classification'] == 'Urban']

In [35]:
kchs_variables = pd.read_csv(paths.PROCESSED_DATA / 'td_kchs_2021_filtered.csv').columns.tolist()
ibs_variables = pd.read_csv(paths.PROCESSED_DATA / 'td_ibs_2015_filtered.csv').columns.tolist()

kchs_variables = [var for var in kchs_variables if var not in ['hhid', 'urban_rural_classification']]
ibs_variables = [var for var in ibs_variables if var not in ['unique_id', 'urban_rural_classification']]

## Run Regression on Exclusion 

In [36]:
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'insig'

def run_exclusion_regression(df: pd.DataFrame, feature_cols: list, label: str):
    """
    Runs a logistic regression where the outcome is exclusion error (actually poor
    but predicted not poor). Returns coefficients and average marginal effects.
    """
    y = df['excluded'].astype(int)

    X = df[feature_cols].copy()
    X = X.drop(columns=['excluded'], errors='ignore')

    # Drop columns with zero variance
    X = X.loc[:, X.nunique() > 1]

    # Drop columns with any nulls
    X = X.dropna(axis=1)

    # Add constant 
    X = sm.add_constant(X)

    # Train model 
    model = sm.OLS(y, X)
    result = model.fit()

    # Coefficients table
    coef_summary = result.summary2().tables[1]

    # Get * sig 
    coef_summary['sig'] = coef_summary['P>|t|'].apply(sig_stars)

    # Calculate percentage points column 
    coef_summary['percentage_points'] = coef_summary['Coef.'] * 100
    
    return coef_summary

kchs_filtered = kchs_df[kchs_variables + ['excluded']]
ibs_filtered  = ibs_df[ibs_variables  + ['excluded']]

kchs_result = run_exclusion_regression(kchs_filtered, kchs_variables, label='KCHS 2021')
ibs_result  = run_exclusion_regression(ibs_filtered,  ibs_variables,  label='IBS 2015')

kchs_result

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975],sig,percentage_points
const,3.996888,0.293272,13.628624,3.674906e-41,3.421870,4.571906,***,399.688794
household_size,-0.006243,0.005594,-1.115961,2.645221e-01,-0.017212,0.004726,insig,-0.624318
mm_age,-0.083377,0.022850,-3.648855,2.675917e-04,-0.128179,-0.038574,***,-8.337653
prop_over65,-0.078510,0.042960,-1.827503,6.771664e-02,-0.162742,0.005722,insig,-7.850985
prop_under5,0.120084,0.044513,2.697746,7.017469e-03,0.032808,0.207361,**,12.008433
...,...,...,...,...,...,...,...,...
occupation_housekeeper,-0.046531,0.053110,-0.876109,3.810364e-01,-0.150664,0.057603,insig,-4.653053
occupation_housewife,-0.066353,0.035633,-1.862143,6.267401e-02,-0.136219,0.003512,insig,-6.635342
occupation_private_sector_worker,0.007212,0.037652,0.191534,8.481190e-01,-0.066612,0.081036,insig,0.721162
occupation_public_sector_worker,-0.036720,0.036623,-1.002642,3.161087e-01,-0.108528,0.035087,insig,-3.672024


In [37]:
kchs_result.to_csv(paths.RESULTS / 'robustness' / 'kchs_exclusion_regression.csv')
ibs_result.to_csv(paths.RESULTS / 'robustness' / 'ibs_exclusion_regression.csv')